# ハミルトニアンシミュレーションの回路資源比較

Trotter–Suzuki、QSVT、multiproduct formula (MPF) を、同じ Pauli 和と誤差予算の下で比較します。小規模系では実際の statevector、大規模系では密行列を作らないゲート分解モデルを使います。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from hamiltonian_resources import (
    BenchmarkConfig, benchmark_scaling, build_multiproduct_circuit,
    build_qsvt_circuit, build_trotter_circuit, compare_with_exact,
    estimate_qsvt_degree, transverse_field_ising,
)
pd.set_option("display.max_columns", None)

## 1. ハミルトニアン入力

例として $H=-J\sum_i Z_iZ_{i+1}-h\sum_iX_i$ を使います。Qiskit の Pauli label は右端が qubit 0 です。任意の Pauli 和は `PauliHamiltonian.from_terms` で渡せます。

In [ ]:
H = transverse_field_ising(3, coupling=1.0, field=0.7)
print(H.name, H.terms)
print(f"terms={H.term_count}, alpha=sum|h_j|={H.alpha:.3f}")

## 2. 実回路を構成する

下の関数はいずれも Hamiltonian の密行列を指数化しません。QSVT の数値的に正しい Hamiltonian-simulation 位相は `synthesize_hamsim_phases` で cosine / sine ごとに生成します。ここでは回路構造を見やすくするため短い例示位相列を使います。

In [ ]:
time = 0.2
trotter_circuit = build_trotter_circuit(H, time, reps=2, order=2)
mpf_circuit = build_multiproduct_circuit(H, time, exponents=(1, 2), segments=1)
qsvt_circuit = build_qsvt_circuit(H, phases=[0.1, 0.2, -0.3, 0.2, 0.1])

display(trotter_circuit.draw(output="mpl", fold=30))
print("MPF metadata:", mpf_circuit.metadata)
print("QSVT metadata:", qsvt_circuit.metadata)

## 3. 小規模系で厳密解と比較

MPF は branch=0 への postselection 後の fidelity と成功確率を別々に表示します。

In [ ]:
validation = []
for reps in (1, 2, 4, 8):
    row = compare_with_exact(H, time, method="trotter", reps=reps, trotter_order=2)
    row["segments"] = reps
    validation.append(row)
for reps in (1, 2, 4):
    row = compare_with_exact(H, time, method="multiproduct", reps=reps, mpf_exponents=(1, 2))
    row["segments"] = reps
    validation.append(row)
pd.DataFrame(validation)

## 4. 固定誤差でシステムサイズを拡大

`transpile_circuits=False` は回路をシミュレーションせず、明示された分解モデルから数えます。`parameter` は Trotter の反復数、MPF の segment 数、QSVT の次数です。T 数には任意角回転の近似合成コストを含めます。

In [ ]:
config = BenchmarkConfig(
    time=0.5, target_error=1e-3, synthesis_error_fraction=0.1,
    trotter_order=2, mpf_exponents=(1, 2, 4),
)
sizes = list(range(2, 31, 2))
resources = benchmark_scaling(
    sizes, lambda n: transverse_field_ising(n, field=0.7), config,
    transpile_circuits=False,
)
resources.head(9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for algorithm, group in resources.groupby("algorithm"):
    axes[0].plot(group.system_size, group.t_count, marker="o", label=algorithm)
    axes[1].plot(group.system_size, group.cnot_count, marker="o", label=algorithm)
for ax, title in zip(axes, ("Estimated T count", "Estimated CNOT count")):
    ax.set_yscale("log")
    ax.set_xlabel("system qubits")
    ax.set_ylabel("gate count")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()

## 5. 小規模で解析モデルを transpile 値に校正

実回路の多重制御分解は急速に大きくなります。まず緩い誤差・2 qubit で試し、その後に範囲を広げてください。

In [ ]:
small_config = BenchmarkConfig(time=0.05, target_error=0.2, mpf_exponents=(1, 2))
compiled = benchmark_scaling(
    [2], transverse_field_ising, small_config, transpile_circuits=True
)
compiled[["algorithm", "num_qubits", "t_count", "cnot_count", "depth"]]

### 解釈上の注意

- QSVT の PREPARE/SELECT コストは Hamiltonian の入力モデルに強く依存します。ここでは一般の Pauli-LCU を公平に数えています。
- MPF の postselection / amplitude amplification コストは成功確率とともに評価してください。
- `choose_parameters` は共通の保守的 error proxy です。論文レベルの比較では、対象 Hamiltonian の交換関係を含む厳密上界または小規模校正値に置き換えてください。
- QSVT の完全な $e^{-iHt}$ は cosine と sine の二つの definite-parity 列の coherent LCU を要します。`build_hamiltonian_qsvt_circuit` がこの結合回路を構成します。